# Colab T4 실험
런타임을 T4 GPU로 설정하고 저장소와 데이터 경로를 바꾸세요. 숫자는 예비 실험용이며 본 실험 전 공통 확정합니다. 체크포인트는 Drive에 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Drive에 코드 폴더와 LOL 데이터를 준비한 후 경로 수정
from pathlib import Path
import shutil
PROJECT = Path('/content/drive/MyDrive/Image_processing_final_Project')
DATA_SOURCE = Path('/content/drive/MyDrive/data/LOL')
DATA = Path('/content/LOL')
assert PROJECT.is_dir() and DATA_SOURCE.is_dir(), 'Drive 경로를 확인하세요'
shutil.copytree(DATA_SOURCE, DATA, dirs_exist_ok=True)
%cd $PROJECT
%pip install -r requirements.txt


In [ ]:
# TASK-003: Colab T4 GPU 및 환경 버전 확인
import platform, subprocess, sys
import numpy, PIL, torch

assert torch.cuda.is_available(), 'GPU 런타임(T4)을 설정하세요: 런타임 > 런타임 유형 변경 > T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
assert 'T4' in gpu_name.upper(), f'T4 GPU가 아닙니다: {gpu_name}'
print('Python:', sys.version.split()[0], f'({platform.platform()})')
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('numpy:', numpy.__version__, '| Pillow:', PIL.__version__)
print(subprocess.check_output(['nvidia-smi'], text=True))


In [ ]:
import subprocess, sys, torch
assert torch.cuda.is_available(), 'GPU 런타임을 설정하세요'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name.upper(), f'T4 GPU가 아닙니다: {gpu_name}'
print(gpu_name)
RUNS = '/content/drive/MyDrive/lowlight_runs'
EPOCHS, CROP, BATCH = 10, 128, 8  # 예비 측정 후 공통 확정
SEED, SPLIT_SEED = 0, 0
common = ['--low-dir',str(DATA/'our485/low'),'--high-dir',str(DATA/'our485/high'),
          '--test-low-dir',str(DATA/'eval15/low'),'--test-high-dir',str(DATA/'eval15/high'),
          '--epochs',str(EPOCHS),'--lr','0.0001','--crop-size',str(CROP),'--batch-size',str(BATCH),
          '--seed',str(SEED),'--split-seed',str(SPLIT_SEED),'--max_hours','3','--output',RUNS]


In [ ]:
# 우선 baseline을 실행하여 시간을 확인하세요. 재개 시 같은 명령에 --resume 추가
subprocess.run([sys.executable,'train.py',*common,'--mode','none'],check=True)


In [ ]:
# 공통 설정 확정 후 실행. 설정을 바꾼 예비 실험과 본 실험은 RUNS 경로를 분리하세요.
for mode in ['bright','noise','both']:
    subprocess.run([sys.executable,'train.py',*common,'--mode',mode],check=True)


In [ ]:
# 튜닝에는 val만 사용
subprocess.run([sys.executable,'eval.py','--runs',RUNS,'--split','val',
                '--output','/content/drive/MyDrive/lowlight_evaluation'],check=True)


In [ ]:
# 설정 확정 후 최종 test. 기본 모델도 같은 조건으로 직접 학습되어 있어야 합니다.
subprocess.run([sys.executable,'eval.py','--runs',RUNS,'--split','test',
                '--output','/content/drive/MyDrive/lowlight_evaluation'],check=True)
